In [2]:
import csv
import os
import pickle

import matplotlib.pyplot as plt
import numpy as np
import scienceplots
from matplotlib.colors import LogNorm
from matplotlib.ticker import MaxNLocator
from scipy.stats import gaussian_kde, wasserstein_distance
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

import os
os.chdir("../")
print(os.getcwd())

from src.Eval import Eval

plt.style.use(['science', 'grid', 'no-latex'])

C:\Users\ismae\Desktop\PhD_codes\SDE_latent


In [3]:
torch.cuda.empty_cache()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
exp_dir ="Your_model_path/Trained_models/"
exp_name = "sl20_ph40_obs64_attblks2_AE_KS"
model = Eval(exp_dir, exp_name)

In [4]:
model.load_weights(min_train_loss = True)

In [ ]:
kolmo_flow ="Your_data_path/KS_10_ens_nu0.8_512.npy"
Phi_test = np.load(kolmo_flow) 
Phi_test = torch.tensor(Phi_test, dtype = torch.float32).to(torch.device("cuda"))
N, T, D = Phi_test.shape

nu = 0.8
L = 64
nx = 512

# time parameters
t0 = 0
dt = 0.25
nt = dt*T

print(f'{N} trajectories') 
print(f'{T} snapshots') 
print(f'{D} x-dimensions') 

In [6]:
def evaluate_ensemble_extended(ensemble, ground_truth):
    """
    Evaluates ensemble predictions.
    
    Returns:
        MSE of the mean flow, PICP at 1 and 3 sigmas, and MPIW.
    """
    N, T, D = ensemble.shape
    

    mean_flow = np.mean(ensemble, axis=0)
    
    rrmse_mean_flow = np.sqrt(np.sum((mean_flow - ground_truth)**2))/np.sqrt(np.sum(ground_truth**2))
    mse_mean_flow = np.mean((mean_flow - ground_truth)**2)
    

    def get_coverage(low_p, high_p):

        lower = np.percentile(ensemble, low_p, axis=0)
        upper = np.percentile(ensemble, high_p, axis=0)
        
        coverage_mask = (ground_truth >= lower) & (ground_truth <= upper)
        
        picp = np.mean(coverage_mask)
        mpiw = np.mean(upper - lower)
        
        return picp, mpiw

    # 1 Sigma approx (68.27% area)
    picp_1s, mpiw_1s = get_coverage(15.85, 84.15)
    
    # 3 Sigma approx (99.73% area)
    picp_3s, mpiw_3s = get_coverage(0.135, 99.865)

    return {
        "rrmse_mean_flow (%)": np.round(rrmse_mean_flow * 100, 2),
        "mse_mean_flow": np.round(mse_mean_flow,3),
        "picp_1sigma (%)": np.round(picp_1s * 100, 2),
        "picp_3sigma (%)": np.round(picp_3s * 100, 2),
        "mpiw_1sigma": np.round(mpiw_1s, 3),
        "mpiw_3sigma": np.round(mpiw_3s, 3)
    }

In [7]:
def moving_average_ensemble(data, window_size=20):
    """
    Performs a moving average on the time dimension (axis 1).
    
    Parameters:
    - data: array of shape [ensembles, T, D]
    - window_size: integer, the size of the smoothing window
    
    Returns:
    - smoothed_data: array of same shape as input
    """

    kernel = np.ones(window_size) / window_size
    

    pad_width = window_size // 2
    padded_data = np.pad(data, ((0,0), (pad_width, pad_width -1), (0,0)), mode='edge')
    
    def smooth_1d(line):
        return np.convolve(line, kernel, mode='valid')

    return np.apply_along_axis(smooth_1d, axis=1, arr=padded_data)

In [ ]:

autoencoder = model.autoencoder_model 
seq_len = model.seq_len
train_size = model.train_size
dt = 0.25

#---------------------------------------Editable parameters-----------------------------------------#
ens = 15 #Ensemble size (about 20 seconds per ensmeble member so ens = 3 -> 1 minute) 
timesteps = 1200   #Inference time 
index = 256    #x coordinate to visualize
trajectory = 0  #Trajectory index from which the initial condition the model starts from
#---------------------------------------------------------------------------------------------------#

fig, ax = plt.subplots(figsize=(6, 5))

model.model.eval()

ens_traj = []

for j in tqdm(range(ens)):
    
    initial_condition = Phi_test[trajectory, :1].to(device)
    Phi_tp = torch.repeat_interleave(initial_condition, seq_len, dim=0)

    if autoencoder == 'CNN' or autoencoder == 'AE' : 
        x_lat  = model.model.autoencoder.encode(Phi_tp)
        
    else : 
        x_lat, mu_lat, log_var = model.model.autoencoder.encode(Phi_tp)

    x = x_lat.unsqueeze(0)
    traj = []

    for t in range(timesteps):
        res_mu_tp1, diffusion, res_x_tp1 = model.model.transformer(x)
        x_tp1 = res_x_tp1 + x[:, -1, :]
        mu_tp1 = res_mu_tp1 + x[:, -1, :]
        Phi_tp1 = model.model.autoencoder.recover(mu_tp1)
        traj.append(Phi_tp1.detach().cpu())
        x = torch.cat([x, x_tp1.unsqueeze(1)], dim=1)[:, 1:, :]

    ens_traj.append(torch.stack(traj))
    torch.cuda.empty_cache()


ens_traj = torch.stack(ens_traj)  # [ens, T, 1, Phi, Nx, Ny]
ens_traj_np = ens_traj[:, :, 0].detach().cpu().numpy()

dict_eval_train = evaluate_ensemble_extended(ens_traj_np[:, :int(train_size*T)], Phi_test[trajectory, :int(train_size*T)].detach().cpu().numpy())
print(dict_eval_train)
dict_eval_test = evaluate_ensemble_extended(ens_traj_np[:, int(train_size*T):T], Phi_test[trajectory, int(train_size*T):T].detach().cpu().numpy())
print(dict_eval_test)

ens_traj_np = moving_average_ensemble(ens_traj_np, window_size=20)

std = np.std(ens_traj_np, axis=0)
mean = np.mean(ens_traj_np, axis=0)
time = np.arange(timesteps) * dt

ax.axvspan(int(train_size * T)*dt, 1000*dt, color='grey', alpha=0.2, label='_nolegend_')
ax.axvspan(T*dt, timesteps*dt, color='grey', alpha=0.4, label='_nolegend_')
ax.axvline(int(train_size * T)*dt, color='black', alpha=1, linestyle = '--', linewidth = 0.3, label='_nolegend_')
ax.axvline(T*dt, color='black', alpha=1, linestyle = '--', linewidth = 0.3, label='_nolegend_')

ax.fill_between(
    time, 
    mean[:, index] - 3*std[:, index],
    mean[:, index] + 3*std[:, index],
    color="#1f77b4", alpha=0.25, label=r"$\pm 3\sigma$"
)

ax.fill_between(
    time, 
    mean[:, index] - std[:, index],
    mean[:, index] + std[:, index],
    color="#ff7f0e", alpha=0.25, label=r"$\pm 1\sigma$"
)


true_traj = Phi_test[trajectory, :, index].detach().cpu().numpy()

ax.plot(
    np.arange(len(true_traj)) * dt, 
    true_traj,
    color="black", linestyle="--", linewidth=1.2, label="Ground Truth"
)

ax.plot(
    time, 
    mean[:, index],
    color="red", label="Mean Forecast", linewidth = 1
)


tick_pos = [400*dt, 900*dt, 1100*dt]
tick_labels = [r'$T_{train}$', r'$T_{test}$', 'Forecast']
ax.set_xticks(tick_pos)
ax.set_xticklabels(tick_labels)

ax.set_title(f"Trajectory {trajectory+1}", fontsize=10)
ax.grid(True, linestyle=':', alpha=0.6)
    
fig.text(0.5, 0.01, "Temporal Domain", ha="center")
fig.text(0.01, 0.5, r"$u_{x_i}(t)$", va="center", rotation="vertical")

handles, labels = ax.get_legend_handles_labels()
by_label = dict(zip(labels, handles))
fig.legend(by_label.values(), by_label.keys(), loc="upper center", ncol=5, frameon=False)

plt.tight_layout(rect=[0.03, 0.03, 1, 0.95])
plt.show()

 67%|██████████████████████████████████████████████████████▋                           | 10/15 [02:41<01:20, 16.15s/it]

In [ ]:
def spectral_derivatives(u, L=64.0):
    """
    u: (T, N) real array, periodic in x
    L: domain length
    returns ux, uxx with same shape as u
    """
    u = np.asarray(u)
    T, N = u.shape

    k = 2*np.pi * np.fft.fftfreq(N, d=L/N)  # wavenumbers
    ik = 1j * k
    k2 = k**2

    U = np.fft.fft(u, axis=1)               # (T, N)
    UX  = np.fft.ifft(ik * U, axis=1).real
    UXX = np.fft.ifft(-k2 * U, axis=1).real
    return UX, UXX


def attractor_density_plot(u, L=64.0, bins=350, xlim=None, ylim=None, log_density=False, ax=None, title=None):
    """
    Builds a 2D density of (ux, uxx) over all (t, x) samples.
    """
    ux, uxx = spectral_derivatives(u, L=L)

    X = ux.ravel()
    Y = uxx.ravel()

    H, xedges, yedges = np.histogram2d(
        X, Y, bins=bins
    )

    return H, xedges, yedges

In [ ]:
plt.rcParams.update({
    "font.size": 18,            # Crisp baseline size
    "axes.labelsize": 24,       # Large, paper-ready axis labels
    "xtick.labelsize": 32,      # Easily readable tick values
    "ytick.labelsize": 32,
    "figure.titlesize": 24
})

plt.style.use(['science', 'no-latex'])
levels = np.arange(-3, 3, 0.01)

x = Phi_test[trajectory].detach().cpu().numpy()
x_hat_ens = ens_traj_np

ens_idx = [0, 1, 2] 
nx = x.shape[1]
xx_ref = np.linspace(start=0, stop=L, num=nx)


xtick_pos = [0, 32, 64]
xtick_labels = ['0', r'$\frac{L}{2}$', '$L$']
ytick_pos_true_data = [400, 900]
ytick_labels_true_data = [r'$T_{train}$', r'$T_{test}$']
ytick_pos_infered_data = [400, 900, 1100]
ytick_labels_infered_data = [r'$T_{train}$',  r'$T_{test}$',  r'Forecast']


fig, axes = plt.subplots(1, 4, figsize=(18, 7), layout='constrained', sharey=True)

xx_ref = np.linspace(start=0, stop=L, num=nx)
tt_ref = np.arange(x.shape[0])

cf0 = axes[0].contourf(xx_ref, tt_ref, x, levels=levels, cmap='bwr', extend='both')
axes[0].set_xticks(xtick_pos)
axes[0].set_xticklabels(xtick_labels)
axes[0].set_yticks(ytick_pos_true_data)
axes[0].set_yticklabels(ytick_labels_true_data)
axes[0].set_title(r'$u(x,t)$', fontsize = 24)


axes[0].axhline(y=int(train_size*T), color='black', linestyle='--', linewidth=1.5, alpha=0.7)
axes[0].axhline(y=T, color='black', linestyle='--', linewidth=1.5, alpha=0.7)
axes[0].axhspan(int(train_size*T), T, color='gray', alpha=0.30)


xx_ens = np.linspace(start=0, stop=L, num=nx)
tt_ens = np.arange(T_inference)

for i, member_idx in enumerate(ens_idx):
    ax = axes[i + 1]
    cf = ax.contourf(xx_ens, tt_ens, x_hat_ens[member_idx], levels=levels, cmap='bwr', extend='both')
    
    ax.set_xticks(xtick_pos)
    ax.set_xticklabels(xtick_labels)
    
    ax.axhline(y=int(train_size*T), color='black', linestyle='--', linewidth=1.5, alpha=0.7)
    ax.axhline(y=T, color='black', linestyle='--', linewidth=1.5, alpha=0.7)
    
    ax.axhspan(int(train_size*T), T, color='gray', alpha=0.30)
    ax.axhspan(T, timesteps, color='gray', alpha=0.50)
    ax.set_title(rf'$\hat{{u}}_{{{i}}}(x,t)$', fontsize=24)


for ax in axes:
    ax.set_xlabel("x", labelpad=10)
    ax.set_ylim(0, timesteps)
    ax.tick_params(axis='both', which='major', labelsize=18)

axes[0].set_ylabel("t", labelpad=10)
axes[0].set_yticks(ytick_pos_infered_data)
axes[0].set_yticklabels(ytick_labels_infered_data)

cbar = fig.colorbar(cf0, ax=axes.ravel().tolist(), orientation='vertical', shrink=1)
cbar.set_label('u', size=24, labelpad=15)
cbar.ax.tick_params(labelsize=16)

plt.show()

In [ ]:
truth = Phi_test[trajectory].detach().cpu().numpy()
T_truth = truth.shape[0]
nx = truth.shape[1]

mean_pred = mean[:T_truth] 
std_pred = std[:T_truth] 
error = (mean_pred - truth)**2


xtick_pos = [0, 32, 64]
xtick_labels = ['0', r'$\frac{L}{2}$', '$L$']
ytick_pos_infered_data = [400, 900]
ytick_labels_infered_data = [r'$T_{train}$', r'$T_{test}$']


fig, axes = plt.subplots(1, 4, figsize=(18, 6), layout='constrained', sharey=True)
xx_ref = np.linspace(start=0, stop=L, num=nx)
tt_ref = np.arange(T_truth)


cf0 = axes[0].contourf(xx_ref, tt_ref, truth, levels=levels, cmap='bwr', extend='both')
axes[0].set_title(r'$u(x,t)$', fontsize=24)


cf1 = axes[1].contourf(xx_ref, tt_ref, mean_pred, levels=levels, cmap='bwr', extend='both')
axes[1].set_title(r'$\bar{\hat{u}}(x,t)$', fontsize=24)


var_levels = np.arange(0, (std_pred**2).max(), 0.005) if (std_pred**2).max() > 0 else 10
cf2 = axes[2].contourf(xx_ref, tt_ref, std_pred**2, levels=var_levels, cmap='twilight', extend='both')
axes[2].set_title(r'$\sigma^2_{\hat{u}}(x,t)$', fontsize=24)


err_levels = np.arange(0, error.max(), 0.005) if error.max() > 0 else 10
cf3 = axes[3].contourf(xx_ref, tt_ref, error, levels=err_levels, cmap='inferno', extend='both')
axes[3].set_title(r'$|| \bar{\hat{u}} - u ||_2$', fontsize=24)

for ax in axes:
    ax.set_xticks(xtick_pos)
    ax.set_xticklabels(xtick_labels)
    ax.set_xlabel("x", labelpad=10)
    ax.set_ylim(0, T)
    ax.tick_params(axis='both', which='major', labelsize=18)
    
    ax.axhline(y=int(train_size*T), color='black', linestyle='--', linewidth=1.5, alpha=0.7)
    ax.axhline(y=T, color='black', linestyle='--', linewidth=1.5, alpha=0.7)
    ax.axhspan(int(train_size*T), T, color='gray', alpha=0.30)


axes[0].set_ylabel("t", labelpad=10)
axes[0].set_yticks(ytick_pos_infered_data)
axes[0].set_yticklabels(ytick_labels_infered_data)

cbar_u = fig.colorbar(cf0, ax=[axes[0], axes[1]], orientation='vertical', shrink=0.85, aspect=25)
cbar_u.set_label('$u$', size=24, labelpad=15)
cbar_u.ax.tick_params(labelsize=16)

cbar_var = fig.colorbar(cf2, ax=axes[2], orientation='vertical', shrink=0.85, aspect=25)
cbar_var.set_label(r'$\sigma^2$', size=24, labelpad=15)
cbar_var.ax.tick_params(labelsize=16)

cbar_err = fig.colorbar(cf3, ax=axes[3], orientation='vertical', shrink=0.85, aspect=25)
cbar_err.set_label('Error', size=24, labelpad=15)
cbar_err.ax.tick_params(labelsize=16)

plt.show()

In [ ]:
ens = 10
#------------------------------------Editable parameters--------------------------------------#
timesteps = 2000
smoothing_filter_window = 20 #If smoothing_filter_window = 1 : very noisy -> smoothing_filter_window = 40 : very smooth
#---------------------------------------------------------------------------------------------#

ens_traj = []

mean_norm = torch.mean(torch.mean(Phi_test**2, dim = 0)) 
initial_condition = torch.mean(Phi_test[:, :1], dim = 0)
initial_norm = torch.mean(initial_condition**2)
alpha = mean_norm/initial_norm
initial_condition = initial_condition*np.sqrt(alpha.item()) #We want to make sure the initial vector has a plausible direction (mean of all directions) and a plausible norm/amount of energy (mean of all norms)

initial_condition = torch.repeat_interleave(initial_condition, seq_len, dim=0)

for j in tqdm(range(ens)):    

    Phi_tp = initial_condition
    
    if autoencoder == 'CNN' or autoencoder == 'AE' : 
        x_lat  = model.model.autoencoder.encode(Phi_tp)
        
    else : 
        x_lat, mu_lat, log_var = model.model.autoencoder.encode(Phi_tp)

    x = x_lat.unsqueeze(0)
    traj = []

    for t in range(timesteps):
        res_mu_tp1, diffusion, res_x_tp1 = model.model.transformer(x)
        x_tp1 = res_x_tp1 + x[:, -1, :]
        mu_tp1 = res_mu_tp1 + x[:, -1, :]
        Phi_tp1 = model.model.autoencoder.recover(mu_tp1)
        traj.append(Phi_tp1.detach().cpu())
        x = torch.cat([x, x_tp1.unsqueeze(1)], dim=1)[:, 1:, :]

    ens_traj.append(torch.stack(traj))
    torch.cuda.empty_cache()

ens_traj= torch.stack(ens_traj) 
ens_traj_np = ens_traj[:, :, 0].numpy()
ens_traj_np_smooth = moving_average_ensemble(ens_traj_np, window_size = smoothing_filter_window)

In [ ]:
plt.style.use(['science', 'grid', 'no-latex'])
plt.rcParams.update({
    "font.size": 18,
    "axes.labelsize": 24,
    "xtick.labelsize": 12,    
    "ytick.labelsize": 12,
    "figure.titlesize": 24
})

x = 256

cmap = plt.colormaps.get_cmap('magma')

fig, ax= plt.subplots(figsize=(8, 3))
    
pred = ens_traj_np_smooth[:, :, x] 
pred_t = np.arange(pred.shape[1]) * 0.25
num_ensembles = pred.shape[0]


for i in range(num_ensembles):
    color = cmap(i / num_ensembles)
    plt.plot(pred_t, pred[i], color=color, alpha=1, linewidth=1)


plt.ylabel(r'$u_{x= \frac{L}{2}}$')
plt.xlabel('Time', fontsize = 16)

plt.tight_layout()
plt.show()

In [ ]:
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 11,     
    "axes.labelsize": 22,  
    "axes.titlesize": 13,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "figure.titlesize": 14
})

fig, ax = plt.subplots(figsize=(6,5))
log_density = True 
u = Phi_test.detach().cpu().numpy().reshape(-1, D)

H, xedges, yedges = attractor_density_plot(
    u, L=L, bins=100, log_density=log_density, title=None
)

norm = LogNorm(vmin=1, vmax=H.max()) if log_density else None

im = plt.imshow(
    H.T,
    origin="lower",
    extent=[xedges[0], xedges[-1], yedges[0], yedges[-1]],
    aspect="auto",
    norm=norm,
    cmap="plasma" 
)

plt.xlabel(r"$u_x$")
plt.ylabel(r"$u_{xx}$")
plt.title(r'True manifold in the $(u_x, u_{xx})$ state-space')
cbar = plt.colorbar(im)
cbar.set_label("Density", size=18, labelpad=10)
cbar.ax.tick_params(labelsize=11)

plt.show()

In [ ]:
fig, axes = plt.subplots(2, ens // 2, figsize=(12, 5.5), sharex=True, sharey=True)
axes = axes.flatten()

log_density = True 

for i in range(ens):
    ax = axes[i]
    u = ens_traj_np_smooth[i]
    
    H, xedges, yedges = attractor_density_plot(
        u, L=L, bins=100, log_density=log_density, title=None
    )

    norm = LogNorm(vmin=1, vmax=H.max()) if log_density else None
    
    im = ax.imshow(
        H.T,
        origin="lower",
        extent=[xedges[0], xedges[-1], yedges[0], yedges[-1]],
        aspect="auto",
        norm=norm,
        cmap="plasma" 
    )

    ax.xaxis.set_major_locator(MaxNLocator(nbins=4))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=4))
    
    if i >= ens // 2:
        ax.set_xlabel(r"$\hat{u}_x$")
    if i % (ens // 2) == 0:
        ax.set_ylabel(r"$\hat{u}_{xx}$")


fig.tight_layout()
fig.subplots_adjust(right=0.88, wspace=0.1, hspace=0.15)
cbar_ax = fig.add_axes([0.91, 0.15, 0.02, 0.7]) # [left, bottom, width, height]
cbar = fig.colorbar(im, cax=cbar_ax)
cbar.set_label("Density", size=18, labelpad=10)
cbar.ax.tick_params(labelsize=11)

plt.show()

In [ ]:
def analyze_predictions(Phi_test, Phi_ms_hat):

    w_dists = []
    pdf_data = []

    true = Phi_test  
    true_full = true.flatten()
    kde_true = gaussian_kde(true_full)
    x_eval = np.linspace(true_full.min(), true_full.max(), 30)
    pdf_true = kde_true(x_eval)

    for i in tqdm(range(10)):
        
        pred = Phi_ms_hat[i, :]
        pred_full = pred.flatten()
        kde_pred = gaussian_kde(pred_full)

        xmin = true_full.min()
        xmax = true_full.max()

        pdf_pred = kde_pred(x_eval)

        w_dist = wasserstein_distance(true_full, pred_full)
        w_dists.append(w_dist)

        pdf_data.append((x_eval, pdf_true, pdf_pred))


    fig, axes = plt.subplots(2, 5, figsize=(16, 5)) 
    axes = axes.flatten()

    for i in range(Phi_ms_hat.shape[0]):
        x_eval, pdf_true, pdf_pred = pdf_data[i]
        axes[i].plot(x_eval, pdf_true, label='True PDF', linewidth=2, color='grey')
        axes[i].plot(x_eval, pdf_pred, label='Predicted PDF', linewidth=2, color='cornflowerblue')
        axes[i].set_title(f"Trajectory {i+1}", fontsize=18)
        axes[i].set_xlabel("Value", fontsize = 18)
        axes[i].set_ylabel("Density", fontsize = 18)

    plt.tight_layout()
    plt.show()

    print("\n=== Metrics Summary ===")
    for i in range(N):
        print(f"Trajectory {i} - W-dist: {w_dists[i]:.4f}")
    print(f"Average Wasserstein Distance: {np.mean(w_dists):.4f}")

    return w_dists

w_dist_U = analyze_predictions(Phi_test.detach().cpu().numpy(), ens_traj_np_smooth)

In [ ]:
#------------------------------------Editable parameters--------------------------------------#
time_lag = 10
x = 511
#---------------------------------------------------------------------------------------------#

U_x = Phi_test[:, time_lag:, x].detach().cpu() 
U_y = Phi_test[:, :-time_lag, x].detach().cpu() 

X = U_x.flatten()
Y = U_y.flatten()

phase_space_delayemb = np.vstack([X, Y])
kde_delayemb = gaussian_kde(phase_space_delayemb)
density_delayemb = kde_delayemb(phase_space_delayemb)

fig, ax = plt.subplots(figsize=(6,5))

sc = ax.scatter(
    X, Y,
    c=density_delayemb,
    s=5,
    cmap="plasma",
    linewidths=0
)

cbar = plt.colorbar(sc, ax=ax)
cbar.set_label('Density', rotation=270, labelpad=20, fontsize = 18)
ax.set_xlabel(r"$u_{t}$", fontsize = 24)
ax.set_ylabel(r"$u_{t-\tau}$", fontsize = 24)
ax.set_aspect("auto")
plt.tight_layout()
plt.show()


In [ ]:
Ntraj = ens_traj.shape[0]
nrows = 2
ncols = int(np.ceil(Ntraj / nrows))

fig, axes = plt.subplots(2, 5, figsize=(12, 6), sharex=True, sharey=True, layout='constrained')
axes = axes.flatten()

for i in range(Ntraj):
    
    X_traj = ens_traj_np_smooth[i, time_lag:, x]
    Y_traj = ens_traj_np_smooth[i, :-time_lag, x]

    ax = axes[i]

    sc = ax.scatter(
        X, Y,
        c=density_delayemb,
        s=20,
        cmap="plasma",
        linewidths=0, 
        alpha = 0.25
    )

    if i == 0 : 
        ax.plot(X_traj, Y_traj, 
                color='black', linewidth=1, label=f'Delay embedding trajectory', alpha=1)
    else : 
        ax.plot(X_traj, Y_traj, 
        color='black', linewidth=1, label='', alpha=1)
        

    ax.set_title(f"Trajectory {i+1}", fontsize=18)
    
    ax.set_xticks([])
    ax.set_yticks([])
    
    if i >= 5: ax.set_xlabel(r"$u_{t}$", fontsize=24)
    if i % 5 == 0: ax.set_ylabel(r"$u_{t-\tau}$", fontsize=24)

cbar = fig.colorbar(sc, ax=axes, orientation='vertical', fraction=0.02, pad=0.02)
cbar.set_label('Density', rotation=270, labelpad=20, fontsize=18)
handles, labels = axes[0].get_legend_handles_labels()

fig.legend(handles, labels, 
           loc='upper center', 
           bbox_to_anchor=(0.5, 0), # Positions it above the title
           ncol=1, 
           fontsize=18)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def autocorr_plot(pred, truth, max_lag=25): 
    """
    Computes and plots temporal correlation decay for multiple trajectories.
    
    Parameters:
    -----------
    pred : np.array [trajs, T, nx, ny] - Predicted spatiotemporal data
    truth : np.array [trajs, T, nx, ny] - Ground truth spatiotemporal data
    max_lag : int - Maximum time steps to shift
    """
    
    num_trajs, T, x = truth.shape
    max_lag = min(max_lag, T - 1)
    

    blue_shades = plt.cm.Blues(np.linspace(0.4, 0.9, num_trajs))
    red_shades = plt.cm.Reds(np.linspace(0.4, 0.9, num_trajs))

    fig, ax = plt.subplots(figsize=(6, 4))
    lags = np.arange(max_lag)*5
    
    for traj_idx in tqdm(range(num_trajs)):
        truth_autocorr = np.zeros(max_lag)
        pred_crosscorr = np.zeros(max_lag)
        
        traj_truth = truth[traj_idx]
        traj_pred = pred[traj_idx]
        
        for i in range(max_lag):
            if i == 0:
                t_ref = traj_truth
                p_ref = traj_pred
                t_delay = traj_truth
                p_delay = traj_pred
            else:
                t_ref = traj_truth[:-5*i, ...] 
                t_delay = traj_truth[5*i:, ...]
                p_ref = traj_pred[:-5*i, ...]
                p_delay = traj_pred[5*i:, ...]
            

            t_ref_flat = t_ref.flatten()
            p_ref_flat = p_ref.flatten()
            t_delay_flat = t_delay.flatten()
            p_delay_flat = p_delay.flatten()
            
            truth_autocorr[i] = np.corrcoef(t_ref_flat, t_delay_flat)[0, 1]
            pred_crosscorr[i] = np.corrcoef(p_ref_flat, p_delay_flat)[0, 1]
            
        truth_label = 'True trajectories autocorrelation' if traj_idx == 0 else ""
        pred_label = 'Generated trajectories autocorrelation' if traj_idx == 0 else ""
        
        # Plot individual trajectory lines
        ax.plot(lags, truth_autocorr, color=blue_shades[traj_idx], alpha=0.85, linewidth=1.5, label=truth_label)
        ax.plot(lags, pred_crosscorr, color=red_shades[traj_idx], alpha=0.85, linewidth=1.5, linestyle='--', label=pred_label)

    # Label adjustments and formatting
    ax.set_xlabel('Time Lag ($t$)', fontsize = 16)
    ax.set_ylabel('Correlation Coefficient', fontsize = 16)

    ax.grid(True, linestyle=':', alpha=0.6)
    ax.legend(loc='lower left', fontsize = 14)
    ax.set_ylim(-1.1, 1.1)
    
    plt.tight_layout()
    plt.show()
    plt.close()

autocorr_plot(ens_traj_np_smooth[:, :1000], Phi_test.detach().cpu())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def autocorr_plot(pred, truth, max_lag=50): 
    """
    Computes and plots temporal correlation decay for multiple trajectories.
    
    Parameters:
    -----------
    pred : np.array [trajs, T, nx, ny] - Predicted spatiotemporal data
    truth : np.array [trajs, T, nx, ny] - Ground truth spatiotemporal data
    max_lag : int - Maximum time steps to shift
    """
    
    num_trajs, T, x = truth.shape
    max_lag = min(max_lag, T - 1)
    
    blue_shades = plt.cm.Purples(np.linspace(0.4, 0.9, num_trajs))
    red_shades = plt.cm.Oranges(np.linspace(0.4, 0.9, num_trajs))

    fig, ax = plt.subplots(figsize=(6, 4))
    lags = np.arange(max_lag)*5
    
    for traj_idx in tqdm(range(num_trajs)):
        truth_autocorr = np.zeros(max_lag)
        pred_crosscorr = np.zeros(max_lag)
        
        traj_truth = truth[traj_idx]
        traj_pred = pred[traj_idx]
        
        for i in range(max_lag):
            if i == 0:
                t_ref = traj_truth
                p_ref = traj_pred
                t_delay = traj_truth
                p_delay = traj_pred
            else:
                t_ref = traj_truth[:, :-5*i]
                t_delay = traj_truth[:, 5*i:]
                p_ref = traj_pred[:, :-5*i]
                p_delay = traj_pred[:, 5*i:]
            

            t_ref_flat = t_ref.flatten()
            p_ref_flat = p_ref.flatten()
            t_delay_flat = t_delay.flatten()
            p_delay_flat = p_delay.flatten()
            
            truth_autocorr[i] = np.corrcoef(t_ref_flat, t_delay_flat)[0, 1]
            pred_crosscorr[i] = np.corrcoef(p_ref_flat, p_delay_flat)[0, 1]
            
        truth_label = 'True trajectories autocorrelation' if traj_idx == 0 else ""
        pred_label = 'Generated trajectories autocorrelation' if traj_idx == 0 else ""
        
        # Plot individual trajectory lines
        ax.plot(lags, truth_autocorr, color=blue_shades[traj_idx], alpha=0.85, linewidth=1.5, label=truth_label)
        ax.plot(lags, pred_crosscorr, color=red_shades[traj_idx], alpha=0.85, linewidth=1.5, linestyle='--', label=pred_label)

    # Label adjustments and formatting
    ax.set_xlabel('Spatil Lag ($x$)', fontsize = 16)
    ax.set_ylabel('Correlation Coefficient', fontsize = 16)

    ax.grid(True, linestyle=':', alpha=0.6)
    ax.legend(loc='lower left', fontsize = 14)
    ax.set_ylim(-1.1, 1.1)
    
    plt.tight_layout()
    plt.show()
    plt.close()

autocorr_plot(ens_traj_np_smooth[:, :1000], Phi_test.detach().cpu())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


def compute_1d_energy_spectrum(U, L=64):
    """Computes the 1D kinetic energy spectrum for a velocity field.

    Parameters:
    -----------
    U : np.ndarray
        Velocity field of shape [T, nx]
    L : float
        Domain length to calculate dx

    Returns:
    --------
    E_average : np.ndarray
        The 1D Energy Spectrum E(k), averaged over time.
    k_wavenumbers : np.ndarray
        The corresponding physical wavenumbers.
    """
    T, nx = U.shape
    dx = L / nx

    U_fluct = U - U.mean(axis=1, keepdims=True)
    k_wavenumbers = 2 * np.pi * np.fft.rfftfreq(nx, d=dx)

    fft_U = np.fft.rfft(U_fluct, axis=1)
    energy_spectrum = (np.abs(fft_U) ** 2) / (nx**2)
    E_average = np.mean(energy_spectrum, axis=0)

    E_average[1:-1] *= 2
    dk = k_wavenumbers[1] - k_wavenumbers[0]

    if dk > 0:
        E_average = E_average / dk

    return E_average, k_wavenumbers


window = 20

E_freq, k_bin_centers = compute_1d_energy_spectrum(
    Phi_test[trajectory].detach().cpu().numpy()
)

fig, ax = plt.subplots(figsize=(8, 6))
num_slices = T // window

for t in range(num_slices):
    start = window * t
    end = window * t + window

    U_slice = ens_traj_np[trajectory, start:end]

    E_freq_pred, _ = compute_1d_energy_spectrum(U_slice)
    alpha_val = 1.0 - (t / num_slices)

    lbl = "Predicted Window Slices" if t == 0 else ""

    ax.loglog(
        k_bin_centers[1:],
        E_freq_pred[1:],
        linestyle="-",
        color="cornflowerblue",
        linewidth=1.5,
        label=lbl,
        alpha=0.2, 
    )


ax.loglog(
    k_bin_centers[1:],
    E_freq[1:],
    marker="^",
    linestyle="-",
    color="black",
    markerfacecolor="orange",
    markersize=7,
    linewidth=1.5,
    label="True Spectrum",
)


plt.grid(True, which="both", linestyle="--", alpha=0.5)
plt.xlabel(r"$k$", fontsize=24)
plt.ylabel(r"$E_k$", fontsize=24)

tick_fontsize = 16
plt.xticks(fontsize=tick_fontsize)
plt.yticks(fontsize=tick_fontsize)

plt.legend(loc="best", fontsize=14, frameon=True)

plt.tight_layout()
plt.show()

In [ ]:
def compute_temporal_energy_spectrum(U, dT):
    
    """Computes the 1D kinetic energy spectrum along the TIME dimension.

    Parameters:
    -----------
    U : np.ndarray
        Velocity field of shape [T, nx]
    dT : float
        Time step interval between samples (crucial for physical frequencies)

    Returns:
    --------
    E_time_avg : np.ndarray
        The 1D Energy Spectrum E(f), averaged over all spatial locations.
    f_frequencies : np.ndarray
        The corresponding physical temporal frequencies.
    """
    T, nx = U.shape


    U_fluct = U - U.mean(axis=0, keepdims=True)
    f_frequencies = np.fft.rfftfreq(T, d=dT)


    fft_U = np.fft.rfft(U_fluct, axis=0)


    energy_spectrum = (np.abs(fft_U) ** 2) / (T**2)
    E_time_avg = np.mean(energy_spectrum, axis=1)

    E_time_avg[1:-1] *= 2

    df = f_frequencies[1] - f_frequencies[0]
    if df > 0:
        E_time_avg = E_time_avg / df

    return E_time_avg, f_frequencies


window = 16

U_true = Phi_test[trajectory].detach().cpu().numpy()
E_time_true, f_bins_true = compute_temporal_energy_spectrum(U_true, dt)

fig, ax = plt.subplots(figsize=(8, 6))

T_total, nx_total = U_true.shape
num_slices = nx_total // window

colors = plt.cm.inferno(np.linspace(0.4, 0.8, num_slices))

for s in range(num_slices):
    start = window * s
    end = window * s + window

    U_slice = ens_traj_np[trajectory, :, start:end]
    E_time_pred, f_bins_pred = compute_temporal_energy_spectrum(U_slice, dt)

    lbl = "Predicted Spatial Sub-domains" if s == 0 else ""

    ax.loglog(
        f_bins_pred[1:],
        E_time_pred[1:],
        linestyle="-",
        color=colors[s],
        linewidth=1.3,
        label=lbl,
        alpha=0.3,
    )

ax.loglog(
    f_bins_true[1:],
    E_time_true[1:],
    marker="o",
    linestyle="-",
    color="black",
    markerfacecolor="yellow",
    markersize=6,
    linewidth=1.8,
    label="True Temporal Spectrum",
)


plt.grid(True, which="both", linestyle="--", alpha=0.5)
plt.xlabel(r"Frequency $f$ (Hz)", fontsize=20)
plt.ylabel(r"$E(f)$", fontsize=20)

tick_fontsize = 14
plt.xticks(fontsize=tick_fontsize)
plt.yticks(fontsize=tick_fontsize)

plt.legend(loc="best", fontsize=13, frameon=True)
plt.tight_layout()
plt.show()